
# Markets & Fuels — Sanity Notebook
This notebook lets you **smoke-test** the end-to-end flow before production:
- Read config
- Fetch **ECB EURUSD** via the *ECB Data Portal API*
- Fetch **Brent (BZ=F)** via Yahoo Finance
- Derive **EUR** prices and add flags
- Write & read **Parquet** output

> Run this in a clean virtual environment with:  
> `pip install pandas requests yfinance pyarrow matplotlib`


In [ ]:

from dataclasses import dataclass, field
from typing import Dict
import os

@dataclass
class Settings:
    http_timeout: int = field(default_factory=lambda: int(os.getenv("HTTP_TIMEOUT", 30)))
    ecb_base: str = field(default_factory=lambda: os.getenv("ECB_API_BASE", "https://data-api.ecb.europa.eu/service"))
    tickers: Dict[str, str] = field(default_factory=dict)

    def __post_init__(self):
        self.tickers.update({
            "brent": os.getenv("YF_TICK_BRNT", "BZ=F"),
            "stoxx": os.getenv("YF_TICK_STOXX", "^STOXX50E"),
            "ibex":  os.getenv("YF_TICK_IBEX", "^IBEX"),
            "cac":   os.getenv("YF_TICK_CAC", "^FCHI"),
            "psi20": os.getenv("YF_TICK_PSI", "^PSI20"),
        })
        self.tickers = {k.lower(): v for k, v in self.tickers.items() if v}

    def get_ticker(self, key: str) -> str:
        k = key.lower()
        if k not in self.tickers or not self.tickers[k]:
            raise KeyError(f"Missing ticker for '{key}'. Set YF_TICK_* env var.")
        return self.tickers[k]

settings = Settings()
settings.ecb_base, settings.get_ticker("brent")


In [ ]:

import pandas as pd
import requests

KEY = "D.USD.EUR.SP00.A"  # daily USD per EUR reference rate

def fetch_eur_usd(start: str, end: str, http_timeout: int = None) -> pd.DataFrame:
    base = settings.ecb_base.rstrip("/")
    url = (
        f"{base}/data/EXR/{KEY}"
        f"?startPeriod={start}&endPeriod={end}"
        f"&detail=dataonly&format=jsondata"
    )
    r = requests.get(url, timeout=http_timeout or settings.http_timeout)
    r.raise_for_status()
    data = r.json()

    time_vals = data["structure"]["dimensions"]["observation"][0]["values"]
    series_dict = data["dataSets"][0]["series"]
    rows = []
    for serie in series_dict.values():
        for idx_str, arr in serie.get("observations", {}).items():
            t = time_vals[int(idx_str)]["id"]
            v = arr[0]
            rows.append({"date": pd.to_datetime(t), "eurusd": float(v)})
    fx = pd.DataFrame(rows).sort_values("date")
    if fx.empty:
        return pd.DataFrame(columns=["date", "eurusd", "usdeur", "source"])
    fx["usdeur"] = 1.0 / fx["eurusd"]
    fx["source"] = "ecb:data-api"
    fx.reset_index(drop=True, inplace=True)
    return fx

fx_demo = fetch_eur_usd("2025-10-01", "2025-10-10")
fx_demo.head(), fx_demo.tail(), len(fx_demo)


In [ ]:

import matplotlib.pyplot as plt

if not fx_demo.empty:
    fx_demo.plot(x="date", y="eurusd", title="EUR/USD (ECB Reference)")
    plt.show()
else:
    print("ECB fetch returned empty dataframe for the given dates.")


In [ ]:

import yfinance as yf

def fetch_brent(start: str, end: str) -> pd.DataFrame:
    sym = settings.get_ticker("brent")
    y = yf.Ticker(sym)
    df = y.history(start=start, end=end, auto_adjust=False)
    if df.empty:
        return pd.DataFrame(columns=["date", "series", "value", "source"])
    out = (
        df[["Close"]]
        .rename(columns={"Close": "value"})
        .reset_index()
        .rename(columns={"Date": "date"})
    )
    out["series"] = "brent_usd_bbl"
    out["source"] = f"yahoo:{sym}"
    return out[["date", "series", "value", "source"]]

br_demo = fetch_brent("2025-10-01", "2025-10-10")
br_demo.head(), len(br_demo)


In [ ]:

def convert_usd_to_eur(prices: pd.DataFrame, fx: pd.DataFrame, out_series: str) -> pd.DataFrame:
    if prices.empty or fx.empty:
        return pd.DataFrame(columns=["date", "series", "value", "source", "is_estimated", "is_ffill"])
    p = prices.rename(columns={"value": "usd"}).copy()
    fx2 = fx[["date", "usdeur"]].copy()
    m = p.merge(fx2, on="date", how="left")
    m["value"] = m["usd"] * m["usdeur"]
    m["series"] = out_series
    m["is_estimated"] = True
    m["is_ffill"] = False
    return m[["date", "series", "value", "source", "is_estimated", "is_ffill"]]

# Flags helper
def add_flags(df: pd.DataFrame, estimated=False, ffill=False) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out["is_estimated"] = bool(estimated)
    if "is_ffill" not in out.columns:
        out["is_ffill"] = bool(ffill)
    return out

br_demo_flags = add_flags(br_demo, estimated=False, ffill=False)
br_eur = convert_usd_to_eur(br_demo_flags[["date","series","value","source"]], fx_demo, "brent_eur_bbl")

combined = pd.concat([br_demo_flags, br_eur], ignore_index=True)
combined.sort_values("date", inplace=True)
combined.head(10), len(combined)


In [ ]:

if not combined.empty:
    pivot = combined.pivot(index="date", columns="series", values="value")
    ax = pivot.plot(title="Brent: USD/bbl vs EUR/bbl")
    plt.show()
else:
    print("No combined data to plot.")


In [ ]:

import pyarrow as pa, pyarrow.parquet as pq
from pathlib import Path

out_path = Path("fuels_daily.parquet")
combined.to_parquet(out_path, index=False)
print("Wrote:", out_path.resolve())

df_check = pd.read_parquet(out_path)
df_check.head(10), df_check.dtypes


In [ ]:

print("ECB base:", settings.ecb_base)
print("HTTP timeout:", settings.http_timeout)
print("Tickers:", settings.tickers)
